## PHASE 1: EXPLORATION AND DATA CLEANING
This section covers loading the data, initial exploration, merging, and handling missing values.

In [397]:
# Import necessary libraries
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Set pandas to display all columns (prevents truncation in wide datasets)
pd.set_option("display.max_columns", None)

# Note: If any of these imports fail, you may need to install them using:
# pip install pandas , numpy , seaborn , or matplotlib

### =========================================
### 1. Exploratory Data Analysis (EDA) and Data Cleaning
### =========================================

In [398]:
df_flights = pd.read_csv("Customer Flight Activity.csv") # Load dataset

display(df_flights.head(2)) # Flights dataset exploration
print("=" * 100)

display(df_flights.tail(2))
print("=" * 100)

display(df_flights.sample(2))
print(f"The flights dataframe has {df_flights.shape[0]} rows and {df_flights.shape[1]} columns")

,Loyalty Number,Year,Month,Flights Booked,Flights with Companions,Total Flights,Distance,Points Accumulated,Points Redeemed,Dollar Cost Points Redeemed
0,100018,2017,1,3,0,3,1521,152.0,0,0
1,100102,2017,1,10,4,14,2030,203.0,0,0


,Loyalty Number,Year,Month,Flights Booked,Flights with Companions,Total Flights,Distance,Points Accumulated,Points Redeemed,Dollar Cost Points Redeemed
405622,999982,2018,12,0,0,0,0,0.0,0,0
405623,999986,2018,12,0,0,0,0,0.0,0,0


,Loyalty Number,Year,Month,Flights Booked,Flights with Companions,Total Flights,Distance,Points Accumulated,Points Redeemed,Dollar Cost Points Redeemed
240118,287385,2018,3,0,0,0,0,0.0,0,0
235677,127590,2018,4,11,0,11,1903,205.2,0,0


The flights dataframe has 405624 rows and 10 columns


In [399]:
df_loyalty = pd.read_csv("Customer Loyalty History.csv")

display(df_loyalty.head(2)) # Loyalty dataset exploration
print("=" * 100)

display(df_loyalty.tail(2))
print("=" * 100)

display(df_loyalty.sample(2))
print(f"The loyalty dataframe has {df_loyalty.shape[0]} rows and {df_loyalty.shape[1]} columns")

,Loyalty Number,Country,Province,City,Postal Code,Gender,Education,Salary,Marital Status,Loyalty Card,CLV,Enrollment Type,Enrollment Year,Enrollment Month,Cancellation Year,Cancellation Month
0,480934,Canada,Ontario,Toronto,M2Z 4K1,Female,Bachelor,83236.0,Married,Star,3839.14,Standard,2016,2,NaN,NaN
1,549612,Canada,Alberta,Edmonton,T3G 6Y6,Male,College,NaN,Divorced,Star,3839.61,Standard,2016,3,NaN,NaN


,Loyalty Number,Country,Province,City,Postal Code,Gender,Education,Salary,Marital Status,Loyalty Card,CLV,Enrollment Type,Enrollment Year,Enrollment Month,Cancellation Year,Cancellation Month
16735,906428,Canada,Yukon,Whitehorse,Y2K 6R0,Male,Bachelor,-57297.0,Married,Star,10018.66,2018 Promotion,2018,4,NaN,NaN
16736,652627,Canada,Manitoba,Winnipeg,R2C 0M5,Female,Bachelor,75049.0,Married,Star,83325.38,Standard,2015,12,2016.0,8.0


,Loyalty Number,Country,Province,City,Postal Code,Gender,Education,Salary,Marital Status,Loyalty Card,CLV,Enrollment Type,Enrollment Year,Enrollment Month,Cancellation Year,Cancellation Month
9539,477654,Canada,British Columbia,Dawson Creek,U5I 4F1,Male,College,NaN,Married,Nova,21423.64,Standard,2018,5,NaN,NaN
12433,572282,Canada,Manitoba,Winnipeg,R3R 3T4,Female,Bachelor,101352.0,Married,Star,3808.12,Standard,2012,11,NaN,NaN


The loyalty dataframe has 16737 rows and 16 columns


In [400]:
# Detailed information for Flights
print("--- Flights Data Summary Table ---")
flights_summary = pd.DataFrame({
    'Data Type': df_flights.dtypes,
    'Non-Null Count': df_flights.count(),
    'Null Count': df_flights.isnull().sum(),
    'Null Percentage (%)': (df_flights.isnull().sum() / len(df_flights)) * 100
}).round(2)
display(flights_summary)

--- Flights Data Summary Table ---


,Data Type,Non-Null Count,Null Count,Null Percentage (%)
Loyalty Number,int64,405624,0,0.0
Year,int64,405624,0,0.0
Month,int64,405624,0,0.0
Flights Booked,int64,405624,0,0.0
Flights with Companions,int64,405624,0,0.0
Total Flights,int64,405624,0,0.0
Distance,int64,405624,0,0.0
Points Accumulated,float64,405624,0,0.0
Points Redeemed,int64,405624,0,0.0
Dollar Cost Points Redeemed,int64,405624,0,0.0


In [401]:
# Detailed information for Loyalty
print("--- Loyalty Data Summary Table ---")
loyalty_summary = pd.DataFrame({
    'Data Type': df_loyalty.dtypes,
    'Non-Null Count': df_loyalty.count(),
    'Null Count': df_loyalty.isnull().sum(),
    'Null Percentage (%)': (df_loyalty.isnull().sum() / len(df_loyalty)) * 100
}).round(2)
display(loyalty_summary)

--- Loyalty Data Summary Table ---


,Data Type,Non-Null Count,Null Count,Null Percentage (%)
Loyalty Number,int64,16737,0,0.00
Country,object,16737,0,0.00
Province,object,16737,0,0.00
City,object,16737,0,0.00
Postal Code,object,16737,0,0.00
Gender,object,16737,0,0.00
Education,object,16737,0,0.00
Salary,float64,12499,4238,25.32
Marital Status,object,16737,0,0.00
Loyalty Card,object,16737,0,0.00


### Observations & Cleaning Strategy

Based on the exploratory data analysis, the following technical strategies have been defined for data cleaning and merging:
 
### **Flights Dataset:**
 * **Identifiers:** Retain `loyalty_number` as `int64` to optimize memory and join performance. As a unique ID, statistical aggregations (mean, median) are not applicable.
 * **Data Integrity & Valid Zeros:** Although there are no null values, we must verify that `loyalty_number`, `year`, and `month` are strictly > 0 to rule out corrupt records. Zeros in flight metrics (e.g., flights booked) represent valid inactive months for users and must be strictly preserved to avoid distorting the temporal behavior.
 * **Temporal Engineering:** Combine the `year` and `month` integer columns into a unified `datetime` feature (`flight_date`) for proper time-series analysis.
 * **Type Consistency:** Cast `points_redeemed` and `dollar_cost_points_redeemed` to `float` to maintain consistency with the `points_accumulated` format.

### **Loyalty Dataset:**
 * **Relational Key:** `loyalty_number` remains an `int64` and will serve as the primary key for joining both dataframes.
 * **Categorical Audit:** String columns will be audited to verify the absence of pseudo-duplicate categories. 
 * **Missing Data & Anomalies (Salary):**  Approximately 25% of `salary` data is missing. We must first identify and neutralize anomalies (e.g., negative salaries) before applying median imputation to avoid skewing the distribution.
 * **Temporal Features & Cancellations (MNAR Strategy):** Over 87% of cancellation data is null, indicating active memberships (Missing Not At Random). To extract this business value, we will create a boolean `is_active` feature. Then, we will generate a unified `cancellation_date` (leaving active users as `NaT`), and finally, drop the original, redundant cancellation year and month columns to optimize memory.


#### Global Schema Standardization (columns)

In [402]:
def standardize_headers(df): # Converts all column names to snake_case format for consistency
    original_cols = df.columns.tolist()
    df.columns = [col.lower().replace(" ", "_") for col in df.columns]
    print(f"Headers standardized: {len(original_cols)} columns processed.")
    return df

In [403]:
# Apply column standardization to both datasets to ensure a uniform schema
df_flights = standardize_headers(df_flights)
df_loyalty = standardize_headers(df_loyalty)

print("\n--- Standardized Flights Headers ---")
print(df_flights.columns.tolist())

print("\n--- Standardized Loyalty Headers ---")
print(df_loyalty.columns.tolist())

Headers standardized: 10 columns processed.
Headers standardized: 16 columns processed.

--- Standardized Flights Headers ---
['loyalty_number', 'year', 'month', 'flights_booked', 'flights_with_companions', 'total_flights', 'distance', 'points_accumulated', 'points_redeemed', 'dollar_cost_points_redeemed']

--- Standardized Loyalty Headers ---
['loyalty_number', 'country', 'province', 'city', 'postal_code', 'gender', 'education', 'salary', 'marital_status', 'loyalty_card', 'clv', 'enrollment_type', 'enrollment_year', 'enrollment_month', 'cancellation_year', 'cancellation_month']


In [404]:
# Combines separate year and month columns into a single unified datetime object.
def create_datetime_feature(df, year_col, month_col, new_name, position=None): # Specifying the insertion position.
    # Create a temporary DataFrame with standardized names for pd.to_datetime
    temp_df = df[[year_col, month_col]].rename(columns={year_col: 'year', month_col: 'month'})
    # Convert to datetime (NaNs will correctly result in NaT - Not a Time)
    new_series = pd.to_datetime(temp_df.assign(day=1)).dt.to_period('M')  # This removes the 'day' component visually while retaining time-series capabilities
    # Insert at specific position if provided, otherwise append to the end
    if position is not None:
        df.insert(position, new_name, new_series)
    else:
        df[new_name] = new_series

    df.drop(columns=[year_col, month_col], inplace=True) # Drop redundant source columns
    return df

In [405]:
df_flights = create_datetime_feature(df_flights, 'year', 'month', 'flight_date', position=2) # Create flight_date

# Create enrollment_date and cancellation_date
df_loyalty = create_datetime_feature(df_loyalty, 'enrollment_year', 'enrollment_month', 'enrollment_date') 
df_loyalty = create_datetime_feature(df_loyalty, 'cancellation_year', 'cancellation_month', 'cancellation_date')

print("Temporal Engineering ✔️: Combine the `year` and `month` integer columns into a unified `datetime`check: created for both datasets.")

Temporal Engineering ✔️: Combine the `year` and `month` integer columns into a unified `datetime`check: created for both datasets.


In [406]:
df_loyalty.sample(5)

,loyalty_number,country,province,city,postal_code,gender,education,salary,marital_status,loyalty_card,clv,enrollment_type,enrollment_date,cancellation_date
3196,461199,Canada,Quebec,Quebec City,G1B 3L5,Male,Bachelor,49588.0,Divorced,Aurora,10972.07,Standard,2015-03,NaT
2506,680878,Canada,Ontario,Sudbury,M5V 1G5,Female,College,NaN,Single,Aurora,8262.63,Standard,2018-08,NaT
6200,646262,Canada,Quebec,Tremblant,H5Y 2S9,Male,Doctor,244110.0,Married,Nova,4893.98,Standard,2014-08,2015-04
9827,717100,Canada,British Columbia,Vancouver,V5R 1W3,Female,College,NaN,Single,Star,1898.68,Standard,2013-03,2018-10
6565,432298,Canada,New Brunswick,Moncton,E1A 2A7,Male,Bachelor,79255.0,Single,Nova,5323.87,Standard,2017-03,NaT


In [407]:
df_flights.sample(5)

,loyalty_number,flight_date,flights_booked,flights_with_companions,total_flights,distance,points_accumulated,points_redeemed,dollar_cost_points_redeemed
29638,775185,2017-02,7,6,13,858,85.0,0,0
360564,402706,2018-10,0,0,0,0,0.0,0,0
117474,955479,2017-07,19,0,19,4370,437.0,0,0
201190,913233,2017-12,0,0,0,0,0.0,0,0
300796,817102,2018-06,0,0,0,0,0.0,0,0


#### Categorical Data Cleaning 


In [408]:
def get_categorical_columns(df): # Returns a list of column names with 'object' data type.
    return df.select_dtypes('object').columns.tolist()

def show_unique_categories(df, columns):
    for col in columns:
        uniques = df[col].unique() # Prints unique values for categorical columns and verifies data consistency. 
        print(f"Unique in '{col}': {uniques}")
        
        # Validation Logic: compare current uniques vs. normalized uniques
        normalized_count = df[col].astype(str).str.lower().str.strip().nunique()
        
        if len(uniques) == normalized_count:
            print(f"Consistency Check: All categories are unique. No casing or spacing duplicates found.")
        else:
            print(f"Consistency Check: Pseudo-duplicates detected. Normalization (.str.lower().str.strip()) would be required.")
        print("-" * 30)


In [409]:
categorical_cols = get_categorical_columns(df_loyalty)

print("--- CATEGORICAL DATA AUDIT ✔️ ---") 
show_unique_categories(df_loyalty, categorical_cols)

--- CATEGORICAL DATA AUDIT ✔️ ---
Unique in 'country': ['Canada']
Consistency Check: All categories are unique. No casing or spacing duplicates found.
------------------------------
Unique in 'province': ['Ontario' 'Alberta' 'British Columbia' 'Quebec' 'Yukon' 'New Brunswick'
 'Manitoba' 'Nova Scotia' 'Saskatchewan' 'Newfoundland'
 'Prince Edward Island']
Consistency Check: All categories are unique. No casing or spacing duplicates found.
------------------------------
Unique in 'city': ['Toronto' 'Edmonton' 'Vancouver' 'Hull' 'Whitehorse' 'Trenton' 'Montreal'
 'Dawson Creek' 'Quebec City' 'Fredericton' 'Ottawa' 'Tremblant' 'Calgary'
 'Thunder Bay' 'Whistler' 'Peace River' 'Winnipeg' 'Sudbury'
 'West Vancouver' 'Halifax' 'London' 'Regina' 'Kelowna' "St. John's"
 'Victoria' 'Kingston' 'Banff' 'Moncton' 'Charlottetown']
Consistency Check: All categories are unique. No casing or spacing duplicates found.
------------------------------
Unique in 'postal_code': ['M2Z 4K1' 'T3G 6Y6' 'V6E 3D9

In [410]:
df_loyalty.sample(2)

,loyalty_number,country,province,city,postal_code,gender,education,salary,marital_status,loyalty_card,clv,enrollment_type,enrollment_date,cancellation_date
4885,757387,Canada,Saskatchewan,Regina,S1J 3C5,Male,Bachelor,77914.0,Divorced,Nova,3230.58,Standard,2015-06,NaT
6761,159942,Canada,Ontario,Toronto,M8Y 4K8,Female,Bachelor,52192.0,Divorced,Nova,5532.63,Standard,2012-08,NaT


#### Numerical Data Cleaning 

In [411]:
# Ensures that every record has a valid identifier and a non-null temporal reference.
invalid_records = df_flights[(df_flights['loyalty_number'] <= 0) | (df_flights['flight_date'].isnull())]
print(f"Data Integrity & Valid Zeros ✔️: {len(invalid_records)} invalid records found (ID or Date <= 0).")

Data Integrity & Valid Zeros ✔️: 0 invalid records found (ID or Date <= 0).


In [412]:
# Type Casting point metrics: Harmonizing integers to floats for consistency
df_flights['points_redeemed'] = df_flights['points_redeemed'].astype(float)
df_flights['dollar_cost_points_redeemed'] = df_flights['dollar_cost_points_redeemed'].astype(float)

print(f"Type Consistency ✔️.\n"
      f"{df_flights[[ 'points_accumulated' , 'points_redeemed', 'dollar_cost_points_redeemed']].dtypes}")

Type Consistency ✔️.
points_accumulated             float64
points_redeemed                float64
dollar_cost_points_redeemed    float64
dtype: object


In [413]:
# Salary Imputation: Neutralize negatives and fill with median
df_loyalty.loc[df_loyalty['salary'] < 0, 'salary'] = np.nan

In [414]:
# Calculate central tendency measures to justify the imputation choice
s_mean = df_loyalty['salary'].mean()
s_median = df_loyalty['salary'].median()
s_mode = df_loyalty['salary'].mode()[0]

# Fill missing values using the median (robust to outliers)
df_loyalty['salary'] = df_loyalty['salary'].fillna(s_median)

In [415]:
# Simplified Verification of Imputation Results
print(f"Missing Data & Anomalies (Salary)✔️:\n"
      f" - Mean: {s_mean:.2f} | Median: {s_median:.2f} | Mode: {s_mode:.2f}\n"
      f" - Remaining Negatives: {len(df_loyalty[df_loyalty['salary'] < 0])}\n"
      f" - Remaining Nulls: {df_loyalty['salary'].isnull().sum()}\n")

Missing Data & Anomalies (Salary)✔️:
 - Mean: 79429.57 | Median: 73510.00 | Mode: 101933.00
 - Remaining Negatives: 0
 - Remaining Nulls: 0



In [438]:
# MNAR Strategy: Interpreting null cancellation dates as Active Memberships.
df_loyalty['is_active'] = df_loyalty['cancellation_date'].isnull()


null_count = df_loyalty['cancellation_date'].isnull().sum() # Confirming nulls match our 'is_active' logic
active_count = df_loyalty['is_active'].sum()

print(f"📅 Active status flag prepared.\n"
      f" - Total nulls in cancellation_date (Raw): {null_count}\n"
      f" - Total customers marked as Active: {active_count}\n"
      f" (Note: Nulls are preserved as 'NaT' because they represent active users)")

print("\n Temporal Features & Cancellations (MNAR Strategy) ✔️")
display(df_loyalty[['loyalty_number', 'enrollment_date', 'cancellation_date', 'is_active']].sample(2))

📅 Active status flag prepared.
 - Total nulls in cancellation_date (Raw): 14670
 - Total customers marked as Active: 14670
 (Note: Nulls are preserved as 'NaT' because they represent active users)

 Temporal Features & Cancellations (MNAR Strategy) ✔️


,loyalty_number,enrollment_date,cancellation_date,is_active
14033,676409,2015-01,NaT,True
8997,760196,2012-05,NaT,True


### Observations & Merging Strategy
* **Join Operation:** Perform a `LEFT JOIN` using the Flights dataset as the left table and Loyalty as the right table on `loyalty_number`. This preserves every transactional record while broadcasting the demographic attributes to each corresponding flight month.

In [444]:
df_merged = pd.merge(df_flights, df_loyalty, on='loyalty_number', how='left')

print(f"Final Merged Dataset - Rows: {df_merged.shape[0]} | Columns: {df_merged.shape[1]}")
print("\nPost-Merge Null Verification:")
# Note: cancellation_date nulls are expected and represent ACTIVE customers.
display(df_merged.isnull().sum()[df_merged.isnull().sum()>0])

Final Merged Dataset - Rows: 405624 | Columns: 23

Post-Merge Null Verification:


cancellation_date    355560
dtype: int64

In [445]:
df_merged.sample(5)

,loyalty_number,flight_date,flights_booked,flights_with_companions,total_flights,distance,points_accumulated,points_redeemed,dollar_cost_points_redeemed,country,province,city,postal_code,gender,education,salary,marital_status,loyalty_card,clv,enrollment_type,enrollment_date,cancellation_date,is_active
286900,976330,2018-05,0,0,0,0,0.0,0.0,0.0,Canada,British Columbia,Victoria,V10 6T5,Male,Bachelor,67684.0,Married,Star,8694.00,Standard,2018-09,NaT,True
97963,815850,2017-06,14,7,21,4851,485.0,0.0,0.0,Canada,Ontario,Toronto,P1W 1K4,Male,Bachelor,57974.0,Single,Aurora,19237.77,Standard,2016-09,NaT,True
70755,268958,2017-05,7,0,7,3038,303.0,0.0,0.0,Canada,British Columbia,Vancouver,V6E 3D9,Female,College,73510.0,Single,Star,4463.00,Standard,2012-08,NaT,True
394466,408347,2018-12,16,7,23,4439,443.0,0.0,0.0,Canada,British Columbia,Victoria,V10 6T5,Male,Master,81333.0,Single,Aurora,6005.85,Standard,2018-07,NaT,True
311527,489394,2018-07,0,0,0,0,0.0,0.0,0.0,Canada,Quebec,Montreal,H2T 2J6,Male,Bachelor,64487.0,Married,Nova,11044.11,Standard,2018-07,NaT,True
